# SAC Irrigation Training - v2.10.0 E2 (Kaggle)

## What E2 tests
Hypothesis: the deadly-triad cascade in v2.7 (Chapter 4 Section 4.4.7) is driven by optimistic-tail amplification of the Q-target.  Replacing the scalar twin-Q critic with a 25-quantile critic and dropping the top 5 quantiles per net at every gradient step truncates the optimistic tail and structurally breaks the positive-feedback loop.

## Key changes from v2.7
| # | Change | Source |
|---|--------|--------|
| 1 | SAC -> TQC (`sb3_contrib.TQC`) | Kuznetsov et al. 2020 |
| 2 | Scalar twin-Q -> 25-quantile twin critic (VDN-per-quantile) | Section 5.2 (n_quantiles=25) |
| 3 | `top_quantiles_to_drop_per_net = 5` (drop 20% of target) | Section 5.2 |
| 4 | `n_step = 1` (no n-step yet - this is E2 only, k=5 alone) | E2 design |
| 5 | All other hyperparameters: v2.7 baseline (ent_coef=0.05 fixed, LR 3e-4 -> 5e-5, etc.) | v4 Section 10.2 |
| 6 | Checkpoint every 25k steps (was 50k) | v2.10 handoff |

## Acceptance criteria (v2.10 handoff Section 6)
- bias_ratio at step 250k within +/- 10% of 1.0 in our negative-reward convention (0.90 <= Q_pred/R_real <= 1.10)
- in-distribution yields (dry, moderate) within 1% of v2.7's published numbers
- spatial action std does not collapse below 0.1 mm

## Diagnostic checks before launch
Cell 3 runs the full unit-test suite plus a 1000-step pilot.  Do NOT proceed to Cell 4 unless all of these pass.

In [ ]:
# Cell 1: Install dependencies and clone the repo.
import subprocess, sys, os

def run(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stderr[-2000:])
        raise RuntimeError(f'Command failed: {cmd}')
    return r.stdout

# NOTE: sb3-contrib version must match the stable-baselines3 version.
# If the install errors out, try sb3-contrib==2.5.0 or sb3-contrib==2.4.0.
run('pip install stable-baselines3==2.6.0 sb3-contrib==2.6.0 gymnasium wandb pytest --quiet')

if os.path.exists('/kaggle/working/thesis'):
    run('cd /kaggle/working && rm -rf thesis')
run('cd /kaggle/working && git clone https://github.com/taratorbati/thesis.git')

# Switch to the v2.10 branch if it exists; fall back to main otherwise.
branch_check = subprocess.run(
    'cd /kaggle/working/thesis && git checkout v2.10 2>/dev/null',
    shell=True, capture_output=True, text=True
)
if branch_check.returncode != 0:
    print('NOTE: v2.10 branch not found on remote yet. Using main.')
    print('      Make sure to commit and push the v2.10 files before running this notebook for real.')

os.chdir('/kaggle/working/thesis')
sys.path.insert(0, '/kaggle/working/thesis')

import torch
print(f'PyTorch:           {torch.__version__}')
print(f'CUDA available:    {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU:               {torch.cuda.get_device_name(0)}')

In [ ]:
# Cell 2: WandB secret (optional; training works without it).
import os
try:
    from kaggle_secrets import UserSecretsClient
    os.environ['WANDB_API_KEY'] = UserSecretsClient().get_secret('WANDB_API_KEY')
    print('OK  WandB API key loaded.')
except Exception as e:
    print(f'NOTE  Could not load WandB key ({e}); training continues without WandB.')

import subprocess
r = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(r.stdout if r.returncode == 0 else 'nvidia-smi failed - no GPU allocated')

In [ ]:
# Cell 3: Pre-training validation (MUST PASS before Cell 4).
#
# Runs:
#   1. existing smoke tests (env, v2.7 policy, obs parity)
#   2. existing factorized-critic tests (v2.7 scalar critic)
#   3. NEW: TQC critic unit tests (test_tqc_critic.py)
#   4. NEW: a 1000-step pilot training run to catch wiring bugs
#
# Abort if any of these fail.

import subprocess, sys

print('Running smoke tests...')
r = subprocess.run(
    [sys.executable, '-m', 'pytest', 'tests/test_rl_smoke.py', '-v', '--tb=short'],
    capture_output=False
)
assert r.returncode == 0, 'SMOKE TESTS FAILED'

print('\nRunning factorized critic tests (v2.7 baseline)...')
r = subprocess.run(
    [sys.executable, '-m', 'pytest', 'tests/test_factorized_critic.py', '-v', '--tb=short'],
    capture_output=False
)
assert r.returncode == 0, 'V2.7 FACTORIZED CRITIC TESTS FAILED'

print('\nRunning TQC critic tests (v2.10)...')
r = subprocess.run(
    [sys.executable, '-m', 'pytest', 'tests/test_tqc_critic.py', '-v', '--tb=short'],
    capture_output=False
)
assert r.returncode == 0, 'V2.10 TQC CRITIC TESTS FAILED'

print('\nRunning 1000-step pilot training (wiring check, ~1 minute)...')
from src.rl.train_v210_e2 import train_tqc_e2
_ = train_tqc_e2(
    seed=999,
    output_dir='/kaggle/working/pilot',
    wandb_project=None,
    total_timesteps=1000,
)
print('\nOK  Pre-flight passed. Proceed to Cell 4.')

In [ ]:
# Cell 4: Full 250k training (TQC v2.10 E2).
#
# Wall time: approximately 2.3-2.5 GPU-hours on Kaggle T4.
#
# Start with SEED=0 for paired comparison against v2.7 seed 0.
# Expand to additional seeds only after E2 seed-0 results are evaluated.

SEED = 0   # CHANGE this for additional seeds (only after seed-0 review)

from src.rl.train_v210_e2 import train_tqc_e2

model = train_tqc_e2(
    seed=SEED,
    output_dir='/kaggle/working/thesis/results/rl',
    wandb_project='sac-irrigation-thesis',
    total_timesteps=250_000,
    enable_per_cell_eval=False,   # TQC per-cell eval not yet wired; leave False
)
print('Training complete.')

In [ ]:
# Cell 5: Copy results to /kaggle/working/ (replay buffer excluded - too large).
import shutil, os

src = f'/kaggle/working/thesis/results/rl/sac_v210_e2_seed{SEED}'
dst = f'/kaggle/working/results_v210_e2_seed{SEED}'

if os.path.exists(dst):
    shutil.rmtree(dst)
shutil.copytree(src, dst, ignore=shutil.ignore_patterns('replay_buffer_latest.pkl'))

print(f'Results copied to {dst}')
print()
for root, _, files in os.walk(dst):
    for f in files:
        p = os.path.join(root, f)
        size = os.path.getsize(p)
        rel  = os.path.relpath(p, dst)
        print(f'  {rel}  ({size/1024:.1f} KB)')

In [ ]:
# Cell 6: Post-training 9-cell evaluation on the test grid.
# Perfect forecast (primary thesis numbers).

import subprocess, sys

model_path = f'/kaggle/working/thesis/results/rl/sac_v210_e2_seed{SEED}/best_model/best_model.zip'

print('Evaluating best_model on 9-cell grid (perfect forecast)...')
r = subprocess.run([
    sys.executable, '-m', 'scripts.experiments.exp_rl_tqc',
    '--model', model_path,
    '--scenario', 'all',
    '--budget',   'all',
    '--forecast', 'perfect',
], capture_output=False)
assert r.returncode == 0, 'PERFECT-FORECAST EVAL FAILED'

print('\nEvaluating best_model on 9-cell grid (noisy forecast, seed=42)...')
r = subprocess.run([
    sys.executable, '-m', 'scripts.experiments.exp_rl_tqc',
    '--model', model_path,
    '--scenario', 'all',
    '--budget',   'all',
    '--forecast', 'noisy',
    '--noise-seed', '42',
], capture_output=False)
if r.returncode != 0:
    print('Noisy-forecast eval failed; proceed with perfect-forecast only.')

In [ ]:
# Cell 7: Quick bias-ratio trajectory plot from CSV (sanity check).
import pandas as pd
import matplotlib.pyplot as plt

csv_path = f'/kaggle/working/thesis/results/rl/sac_v210_e2_seed{SEED}/bias_ratio_log.csv'
df = pd.read_csv(csv_path)
print(df.tail(10))

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(df['step'], df['bias_ratio_mean'], '-o', label='bias_ratio_mean')
ax.axhline(1.0, color='k', linestyle=':', alpha=0.5, label='ideal')
ax.axhline(1.10, color='r', linestyle=':', alpha=0.5, label='+10% threshold')
ax.axhline(0.90, color='r', linestyle=':', alpha=0.5)
ax.set_xlabel('training step')
ax.set_ylabel('bias ratio = Q_pred / R_realised')
ax.set_title(f'v2.10 E2 seed {SEED} - cascade diagnostic')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'/kaggle/working/bias_ratio_seed{SEED}.png', dpi=120)
plt.show()
print('Acceptance: bias_ratio should be in [0.90, 1.10] at step 250k.')